# How Images Become Numbers: Visualizing Pixels as Heatmaps

A digital image is just a grid of numbers. For an 8-bit grayscale image, every
pixel is an integer from **0 (black)** to **255 (white)**. A color image is
usually three such grids stacked together (Red, Green, Blue channels).

This notebook makes that concrete by:
1. Loading a sample image and converting it to a NumPy array of numbers.
2. Showing the whole image as a heatmap (too many pixels to label individually).
3. **Zooming in** on a small patch and showing the heatmap **with the actual
   0-255 values annotated on each cell** (`annot=True`).
4. **Zooming out** again to see how the patch fits into the full picture.
5. Doing the same for a color image, one channel (R/G/B) at a time.

Libraries used: `numpy`, `matplotlib`, `seaborn`, `scikit-image` (for a
built-in sample image, no download needed).

In [ ]:
!pip install scikit-image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from skimage import data, transform

sns.set_theme(style="white")
plt.rcParams["figure.dpi"] = 110

## 1. Load an image and look at the raw numbers

`skimage.data.camera()` ships with scikit-image, so this works with no
internet access and no upload. It's already an 8-bit grayscale image
(dtype `uint8`, values 0-255).

In [ ]:
gray = data.camera()  # 512x512 uint8 grayscale image, values 0-255

print("Shape:", gray.shape)
print("dtype:", gray.dtype)
print("Min / Max pixel value:", gray.min(), "/", gray.max())
print()
print("Top-left 5x5 corner as raw numbers:")
print(gray[:5, :5])

In [ ]:
fig, axs = plt.subplots(ncols=3, figsize=(24, 8))
axs[0].imshow(gray, cmap="gray")
# add a rectangle to the heatmap to indicate the area of the image that is shown


box1 = (275, 300, 275, 300)  # rows 250-275, cols 250-275 -> a 25x25 patch
box_hw = (box1[1] - box1[0], box1[3] - box1[2])

rect = patches.Rectangle(
    (box1[0], box1[2]),
    width=box_hw[1],
    height=box_hw[0],
    linewidth=1,
    edgecolor="red",
    facecolor="none",
)
axs[0].add_patch(rect)


box2 = (10, 15, 12, 17)
sns.heatmap(
    gray[box1[0] : box1[1], box1[2] : box1[3]],
    cmap="gray",
    cbar=False,
    annot=True,
    fmt="d",
    square=True,
    linewidths=0.5,
    ax=axs[1],
    vmin=0,
    vmax=255,
    annot_kws={"fontsize": 3},
)
rect2 = patches.Rectangle(
    (box2[0], box2[2]),
    width=box2[3] - box2[2],
    height=box2[1] - box2[0],
    linewidth=1,
    edgecolor="red",
    facecolor="none",
)
axs[1].add_patch(rect2)
axs[1].set_xticklabels(range(box1[2], box1[3]), fontsize=7)
axs[1].set_yticklabels(range(box1[0], box1[1]), fontsize=7, rotation=0)

print(
    f"end={box1[0] + box2[0]}, start={box1[0] + box2[0] + box2[1]}, end={box1[2] + box2[2]}, start={box1[2] + box2[2] + box2[3]}"
)

sns.heatmap(
    gray[287:292, 285:290],
    cmap="gray",
    cbar=False,
    annot=True,
    fmt="d",
    square=True,
    linewidths=0.5,
    ax=axs[2],
    vmin=0,
    vmax=255,
)


# connect the cornes of the rectangles with lines between the plots
from matplotlib.patches import ConnectionPatch

# rect (axs[0]) -> axs[1]
x0, y0 = box1[0], box1[2]  # top-left corner of rect, axs[0] data coords
x1, y1 = x0 + box_hw[1], y0 + box_hw[0]  # bottom-right corner of rect
h1 = box1[1] - box1[0]  # height of axs[1]'s own data coords

for (xA, yA), (xB, yB) in [((x1, y0), (0, 0)), ((x1, y1), (0, h1))]:
    con = ConnectionPatch(
        xyA=(xA, yA),
        coordsA=axs[0].transData,
        axesA=axs[0],
        xyB=(xB, yB),
        coordsB=axs[1].transData,
        axesB=axs[1],
        color="red",
        linewidth=1,
    )
    fig.add_artist(con)

# rect2 (axs[1]) -> axs[2]
x0b, y0b = box2[0], box2[2]
x1b, y1b = x0b + (box2[3] - box2[2]), y0b + (box2[1] - box2[0])
h2 = box2[1] - box2[0]  # height of axs[2]'s own data coords

for (xA, yA), (xB, yB) in [((x1b, y0b), (0, 0)), ((x1b, y1b), (0, h2))]:
    con = ConnectionPatch(
        xyA=(xA, yA),
        coordsA=axs[1].transData,
        axesA=axs[1],
        xyB=(xB, yB),
        coordsB=axs[2].transData,
        axesB=axs[2],
        color="red",
        linewidth=1,
    )
    fig.add_artist(con)


# add lines to the second plot to indicate the area of the image that is shown
axs[1].axhline(0, color="red", linewidth=1)
axs[1].axhline(50, color="red", linewidth=1)
axs[1].axvline(0, color="red", linewidth=1)
axs[1].axvline(50, color="red", linewidth=1)

## 2. The full image as a heatmap (no annotations)

With 512x512 = 262,144 numbers, printing every value on the plot would be
unreadable — so at this zoom level we just show the numbers *as color*.
That mapping (0 -> black/dark, 255 -> white/light) is exactly what a
grayscale image display already is.

In [ ]:
def show_full_with_zoom_box(img, box, box_color="red", ax=None):
    """Show the full image as a heatmap and draw a rectangle around the
    region we're about to zoom into. box = (row_start, row_end, col_start, col_end)"""

    ax = ax or plt.gca()

    r0, r1, c0, c1 = box
    sns.heatmap(
        img,
        cmap="gray",
        vmin=0,
        vmax=255,
        cbar=True,
        square=True,
        xticklabels=False,
        yticklabels=False,
        ax=ax,
    )
    rect = patches.Rectangle(
        (c0, r0), c1 - c0, r1 - r0, linewidth=2, edgecolor=box_color, facecolor="none"
    )
    ax.add_patch(rect)
    ax.set_title("Full image (each cell = 1 pixel, colored by its 0-255 value)")


# Region we'll zoom into: a small patch near the eye of the photographer
zoom_box = (90, 120, 120, 150)  # rows 60-68, cols 120-128 -> an 8x8 patch
show_full_with_zoom_box(gray, zoom_box)

## 3. Zoom in: an 8x8 patch, with every number visible (`annot=True`)

Now we crop out that small red box and re-draw it much larger, with
`annot=True` so seaborn writes the exact 0-255 value inside every cell.
This is the "machine-readable" view: a computer never sees the picture,
only this grid of integers.

In [ ]:
def show_patch_annotated(img, box, title=None, fmt="d", cmap="gray", ax=None):
    r0, r1, c0, c1 = box
    patch = img[r0:r1, c0:c1]
    ax = ax or plt.gca()
    sns.heatmap(
        patch,
        annot=True,
        fmt=fmt,
        cmap=cmap,
        vmin=0,
        vmax=255,
        cbar=True,
        square=True,
        xticklabels=range(c0, c1),
        yticklabels=range(r0, r1),
        annot_kws={"size": 5},
        ax=ax,
    )
    ax.set_title(title or f"Zoomed in: rows {r0}-{r1}, cols {c0}-{c1} (raw pixel values)")
    ax.set_xlabel("column index")
    ax.set_ylabel("row index")
    plt.show()
    return patch


fig, axs = plt.subplots(ncols=2, figsize=(18, 9))

show_full_with_zoom_box(gray, zoom_box, ax=axs[0])

patch_8x8 = show_patch_annotated(gray, zoom_box, ax=axs[1])

## 4. Zoom in further: a 4x4 corner of that patch

The more we zoom in, the fewer pixels are on screen, but every one is
labeled -- this is the maximum "machine-readable" detail: individual
numbers a neural network or any image-processing algorithm would consume.

In [ ]:
tighter_box = (60, 64, 120, 124)  # 4x4 corner of the previous patch
_ = show_patch_annotated(gray, tighter_box, title="Zoomed in further: a 4x4 pixel block")

## 5. Color images: three grids stacked (R, G, B)

A color image adds a third dimension: for every pixel there are *three*
numbers, one per channel, each still 0-255. Let's load a small color image
and inspect one pixel's three numbers, then zoom into each channel
separately as its own annotated heatmap.

In [ ]:
color_img = data.astronaut()  # 512x512x3 uint8 RGB image
color_img_small = transform.resize(color_img, (128, 128), preserve_range=True).astype(np.uint8)

print("Shape (height, width, channels):", color_img_small.shape)
r, c = 40, 60
print(f"Pixel at row={r}, col={c} -> R,G,B =", color_img_small[r, c])

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(color_img_small)
rect = patches.Rectangle((c - 4, r - 4), 8, 8, linewidth=2, edgecolor="yellow", facecolor="none")
ax.add_patch(rect)
ax.set_title("Color image (each pixel = 3 numbers: R, G, B)")
ax.axis("off")
plt.show()

In [ ]:
def show_channel_patch(img, box, channel_idx, channel_name, cmap, ax=None):
    r0, r1, c0, c1 = box
    patch = img[r0:r1, c0:c1, channel_idx]
    ax = ax or plt.gca()
    sns.heatmap(
        patch,
        annot=True,
        fmt="d",
        cmap=cmap,
        vmin=0,
        vmax=255,
        cbar=False,
        square=True,
        xticklabels=range(c0, c1),
        yticklabels=range(r0, r1),
        annot_kws={"size": 9},
        ax=ax,
    )
    # ax.set_title(f"{channel_name} channel, rows {r0}-{r1}, cols {c0}-{c1}")


color_box = (36, 44, 56, 64)  # 8x8 patch around the pixel we inspected
show_channel_patch(color_img_small, color_box, 0, "Red", "Reds")
show_channel_patch(color_img_small, color_box, 1, "Green", "Greens")
show_channel_patch(color_img_small, color_box, 2, "Blue", "Blues")

In [ ]:
fig, axs = plt.subplots(ncols=3, nrows=2, figsize=(12, 8))

axs[0, 0].imshow(color_img_small, cmap="gray")

rect = patches.Rectangle((c - 4, r - 4), 8, 8, linewidth=2, edgecolor="red", facecolor="none")
axs[0, 0].add_patch(rect)

axs[0, 1].axis("off")
axs[0, 2].imshow(
    color_img_small[color_box[0] : color_box[1], color_box[2] : color_box[3]], cmap="gray"
)
axs[0, 2].set_xticklabels(range(color_box[2], color_box[3]), fontsize=7)
axs[0, 2].set_yticklabels(range(color_box[0], color_box[1]), fontsize=7, rotation=0)


show_channel_patch(color_img_small, color_box, 0, "Red", "Reds", ax=axs[1, 0])
show_channel_patch(color_img_small, color_box, 1, "Green", "Greens", ax=axs[1, 1])
show_channel_patch(color_img_small, color_box, 2, "Blue", "Blues", ax=axs[1, 2])

## Recap

- An image is stored as a NumPy array of integers (0-255 for 8-bit images).
- `plt.imshow` / a heatmap without labels shows those numbers *as color* --
  useful for whole images.
- `sns.heatmap(..., annot=True)` shows the numbers *as text on top of the
  color* -- useful once you zoom in far enough (roughly under ~20x20
  cells) that the numbers stay legible.
- Zooming in = cropping to fewer pixels and turning annotations on.
- Zooming out = looking at a larger crop (or the whole image) and turning
  annotations off, since there isn't room to print every value.
- A color image is simply three of these grids (R, G, B) stacked together,
  one number per channel per pixel.

Try it on your own image: replace `gray = data.camera()` with
`gray = np.array(Image.open("your_file.png").convert("L"))` (using
`from PIL import Image`) and re-run.